In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Import Required Libraries

Let's start by importing all necessary libraries for this workshop.

# XGBoost Workshop: A Comprehensive Guide

In this workshop, we will explore **XGBoost** (eXtreme Gradient Boosting) and learn how to:

1. **Generate random datasets** using scikit-learn
2. **Handle missing values** with different imputation strategies
3. **Encode categorical variables** for machine learning models
4. **Train and evaluate** an XGBoost model
5. **Analyze feature importance** to understand model decisions

This notebook covers both classification and regression tasks, with practical examples and best practices for production-ready pipelines.

## 2. Generate Random Dataset

We'll create a synthetic dataset using scikit-learn's `make_classification`. This dataset will include:
- Numerical features
- Multiple classes
- A specified proportion of missing values

We'll then introduce missing values artificially to demonstrate handling techniques.

In [ ]:
# Generate synthetic classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=15,
    n_informative=10,
    n_redundant=3,
    n_clusters_per_class=2,
    random_state=42,
    class_sep=0.8
)

# Create a DataFrame for easier manipulation
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nTarget distribution:")
print(df['target'].value_counts())

## 3. Handle Missing Values

XGBoost can handle missing values natively, but we'll also demonstrate common imputation strategies. We'll artificially introduce missing values (NaN) and show different approaches:
- **Mean imputation**: Replace with feature mean
- **Median imputation**: Replace with feature median
- **Forward fill**: Use the last valid observation
- **Drop rows**: Remove rows with missing values

In [ ]:
# Create a copy of the dataframe to introduce missing values
df_with_missing = df.copy()

# Introduce missing values (15% of data) randomly
missing_rate = 0.15
for col in feature_names:
    missing_indices = np.random.choice(df_with_missing.index, 
                                      size=int(len(df_with_missing) * missing_rate), 
                                      replace=False)
    df_with_missing.loc[missing_indices, col] = np.nan

print("Missing values count:")
print(df_with_missing.isnull().sum())
print(f"\nTotal missing values: {df_with_missing.isnull().sum().sum()}")
print(f"Percentage of missing data: {(df_with_missing.isnull().sum().sum() / df_with_missing.size * 100):.2f}%")

In [ ]:
# Strategy 1: Mean Imputation
df_mean_imputed = df_with_missing.copy()
df_mean_imputed[feature_names] = df_mean_imputed[feature_names].fillna(df_mean_imputed[feature_names].mean())

print("Strategy 1: Mean Imputation")
print("Missing values after imputation:", df_mean_imputed[feature_names].isnull().sum().sum())
print("First few rows after imputation:")
print(df_mean_imputed.head())

# Strategy 2: Median Imputation
df_median_imputed = df_with_missing.copy()
df_median_imputed[feature_names] = df_median_imputed[feature_names].fillna(df_median_imputed[feature_names].median())

print("\n\nStrategy 2: Median Imputation")
print("Missing values after imputation:", df_median_imputed[feature_names].isnull().sum().sum())

In [ ]:
# Strategy 3: Drop rows with missing values
df_dropped = df_with_missing.dropna()

print("Strategy 3: Drop Rows with Missing Values")
print(f"Original shape: {df_with_missing.shape}")
print(f"Shape after dropping NaN: {df_dropped.shape}")
print(f"Rows dropped: {df_with_missing.shape[0] - df_dropped.shape[0]}")

# Strategy 4: XGBoost Native Handling (it can handle NaN directly)
print("\n\nStrategy 4: XGBoost Native NaN Handling")
print("XGBoost can handle missing values natively without imputation!")
print("It treats missing values as a special value and learns a direction for each split.")

## 4. Encode Categorical Variables

Tree-based models like XGBoost can handle categorical variables, but we need to encode them. We'll demonstrate:
- **Label Encoding**: Convert categories to integers (good for ordinal data)
- **One-Hot Encoding**: Create binary columns for each category (good for nominal data)
- **XGBoost Native Categorical Support**: XGBoost can handle categorical features directly (XGBoost >= 1.5)

In [ ]:
# Add categorical features to the dataset for demonstration
df_with_categorical = df_mean_imputed.copy()

# Create categorical variables
np.random.seed(42)
categories_color = np.random.choice(['Red', 'Green', 'Blue'], size=len(df_with_categorical))
categories_size = np.random.choice(['Small', 'Medium', 'Large'], size=len(df_with_categorical))
categories_season = np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], size=len(df_with_categorical))

df_with_categorical['color'] = categories_color
df_with_categorical['size'] = categories_size
df_with_categorical['season'] = categories_season

print("Dataset with categorical features:")
print(df_with_categorical.head())
print("\nCategorical columns data types:")
print(df_with_categorical[['color', 'size', 'season']].dtypes)

In [ ]:
# Strategy 1: Label Encoding
df_label_encoded = df_with_categorical.copy()
label_encoders = {}

for col in ['color', 'size', 'season']:
    le = LabelEncoder()
    df_label_encoded[col] = le.fit_transform(df_label_encoded[col])
    label_encoders[col] = le
    print(f"Label encoding for {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nDataset after Label Encoding:")
print(df_label_encoded[['color', 'size', 'season']].head(10))
print("\nData types:")
print(df_label_encoded[['color', 'size', 'season']].dtypes)

In [ ]:
# Strategy 2: One-Hot Encoding
df_onehot = df_with_categorical.copy()

# Drop original categorical columns and add one-hot encoded versions
categorical_cols = ['color', 'size', 'season']
df_onehot = pd.get_dummies(df_onehot, columns=categorical_cols, drop_first=True)

print("Dataset after One-Hot Encoding:")
print(df_onehot.head())
print("\nNew shape:", df_onehot.shape)
print("\nNew columns created:")
print([col for col in df_onehot.columns if col not in feature_names + ['target']])

In [ ]:
# Strategy 3: XGBoost Native Categorical Support
print("XGBoost Native Categorical Support:")
print("XGBoost (>=1.5) can handle categorical features natively!")
print("You can specify categorical features with: categorical_features=['color', 'size', 'season']")
print("\nAdvantages:")
print("- No need to one-hot encode before training")
print("- Reduces memory usage for high-cardinality features")
print("- Often provides better performance than manual encoding")
print("- Splits are more interpretable")

## 5. Split Data into Train and Test Sets

We'll prepare three datasets for comparison:
1. **With mean imputation** (traditional approach)
2. **With one-hot encoding** (manual categorical handling)
3. **With label encoding + native categorical support** (XGBoost native)

In [ ]:
# Prepare datasets for modeling

# Dataset 1: Label Encoded (Simple approach)
X_label = df_label_encoded.drop('target', axis=1)
y = df_label_encoded['target']

X_train_label, X_test_label, y_train, y_test = train_test_split(
    X_label, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset 1: Label Encoded")
print(f"Training set shape: {X_train_label.shape}")
print(f"Test set shape: {X_test_label.shape}")
print(f"Target distribution (train): {y_train.value_counts().to_dict()}")
print(f"Target distribution (test): {y_test.value_counts().to_dict()}")

# Dataset 2: One-Hot Encoded
X_onehot = df_onehot.drop('target', axis=1)
y_onehot = df_onehot['target']

X_train_onehot, X_test_onehot, y_train_onehot, y_test_onehot = train_test_split(
    X_onehot, y_onehot, test_size=0.2, random_state=42, stratify=y_onehot
)

print("\n\nDataset 2: One-Hot Encoded")
print(f"Training set shape: {X_train_onehot.shape}")
print(f"Test set shape: {X_test_onehot.shape}")

## 6. Train XGBoost Models

We'll train three XGBoost models with different approaches to categorical and missing value handling.

In [ ]:
# Model 1: With Label Encoding
print("Training Model 1: Label Encoded Data")
model_1 = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    random_state=42,
    verbosity=0
)

model_1.fit(X_train_label, y_train)
print("✓ Model 1 trained successfully\n")

# Model 2: With One-Hot Encoding
print("Training Model 2: One-Hot Encoded Data")
model_2 = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    random_state=42,
    verbosity=0
)

model_2.fit(X_train_onehot, y_train_onehot)
print("✓ Model 2 trained successfully\n")

# Model 3: With Native Categorical Support
print("Training Model 3: Native Categorical Support (Label Encoded)")
model_3 = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    random_state=42,
    enable_categorical=True,
    verbosity=0
)

# Convert categorical columns to categorical dtype for native support
X_train_native = X_train_label.copy()
X_test_native = X_test_label.copy()

categorical_indices = [X_train_native.columns.get_loc(col) for col in ['color', 'size', 'season']]
for col in ['color', 'size', 'season']:
    X_train_native[col] = X_train_native[col].astype('category')
    X_test_native[col] = X_test_native[col].astype('category')

model_3.fit(X_train_native, y_train)
print("✓ Model 3 trained successfully")

## 7. Make Predictions

Generate predictions on the test set using all three trained models.

In [ ]:
# Make predictions

# Model 1 predictions
y_pred_1 = model_1.predict(X_test_label)
y_pred_proba_1 = model_1.predict_proba(X_test_label)

# Model 2 predictions
y_pred_2 = model_2.predict(X_test_onehot)
y_pred_proba_2 = model_2.predict_proba(X_test_onehot)

# Model 3 predictions
y_pred_3 = model_3.predict(X_test_native)
y_pred_proba_3 = model_3.predict_proba(X_test_native)

# Display sample predictions
print("Sample Predictions from Model 1 (Label Encoded):")
print("First 10 predictions:", y_pred_1[:10])
print("\nFirst 10 prediction probabilities (class 0, class 1):")
print(y_pred_proba_1[:10])

print("\n\nComparison of predictions from all three models (first 10 samples):")
comparison_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Model1_Pred': y_pred_1[:10],
    'Model2_Pred': y_pred_2[:10],
    'Model3_Pred': y_pred_3[:10]
})
print(comparison_df)

## 8. Evaluate Model Performance

Calculate and compare performance metrics across all three models.

In [ ]:
# Evaluate models

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# Evaluate all models
results_1 = evaluate_model(y_test, y_pred_1, y_pred_proba_1, "Model 1: Label Encoding")
results_2 = evaluate_model(y_test_onehot, y_pred_2, y_pred_proba_2, "Model 2: One-Hot Encoding")
results_3 = evaluate_model(y_test, y_pred_3, y_pred_proba_3, "Model 3: Native Categorical")

# Compare results
results_comparison = pd.DataFrame([results_1, results_2, results_3])
print("\n\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(results_comparison.to_string(index=False))

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

axes[0].bar(x - width, results_comparison.loc[0, metrics], width, label=results_1['Model'], alpha=0.8)
axes[0].bar(x, results_comparison.loc[1, metrics], width, label=results_2['Model'], alpha=0.8)
axes[0].bar(x + width, results_comparison.loc[2, metrics], width, label=results_3['Model'], alpha=0.8)

axes[0].set_xlabel('Metrics', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.7, 1.0])

# Plot 2: Accuracy comparison
models = results_comparison['Model'].str.extract(r'Model \d+: (.+)')[0].values
accuracies = results_comparison['Accuracy'].values
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = axes[1].barh(models, accuracies, color=colors, alpha=0.8)
axes[1].set_xlabel('Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('Accuracy Comparison Across Models', fontsize=13, fontweight='bold')
axes[1].set_xlim([0.7, 1.0])

# Add value labels on bars
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    axes[1].text(acc - 0.02, i, f'{acc:.4f}', va='center', ha='right', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

print("\nVisualization complete!")

## 9. Feature Importance Analysis

Feature importance shows which features the model relies on most for making predictions. This helps with:
- **Model interpretability**: Understanding what drives predictions
- **Feature selection**: Identifying redundant features
- **Domain validation**: Checking if results align with business knowledge

In [ ]:
# Extract feature importance from models

# Model 1 feature importance
importance_1 = model_1.feature_importances_
features_1 = X_train_label.columns

# Create DataFrame for easier visualization
importance_df_1 = pd.DataFrame({
    'Feature': features_1,
    'Importance': importance_1
}).sort_values('Importance', ascending=False)

print("Model 1: Feature Importance (Top 10)")
print(importance_df_1.head(10).to_string(index=False))

# Model 2 feature importance
importance_2 = model_2.feature_importances_
features_2 = X_train_onehot.columns

importance_df_2 = pd.DataFrame({
    'Feature': features_2,
    'Importance': importance_2
}).sort_values('Importance', ascending=False)

print("\n\nModel 2: Feature Importance (Top 10)")
print(importance_df_2.head(10).to_string(index=False))

# Model 3 feature importance
importance_3 = model_3.feature_importances_
features_3 = X_train_native.columns

importance_df_3 = pd.DataFrame({
    'Feature': features_3,
    'Importance': importance_3
}).sort_values('Importance', ascending=False)

print("\n\nModel 3: Feature Importance (Top 10)")
print(importance_df_3.head(10).to_string(index=False))

In [ ]:
# Visualize feature importance for all models

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Model 1
top_n = 12
importance_df_1.head(top_n).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance', ax=axes[0], color='#1f77b4', legend=False
)
axes[0].set_title('Model 1: Label Encoding\nTop 12 Features', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance', fontsize=10)
axes[0].set_ylabel('')

# Model 2
importance_df_2.head(top_n).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance', ax=axes[1], color='#ff7f0e', legend=False
)
axes[1].set_title('Model 2: One-Hot Encoding\nTop 12 Features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance', fontsize=10)
axes[1].set_ylabel('')

# Model 3
importance_df_3.head(top_n).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance', ax=axes[2], color='#2ca02c', legend=False
)
axes[2].set_title('Model 3: Native Categorical\nTop 12 Features', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Importance', fontsize=10)
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

print("Feature Importance Visualization Complete!")